In [ ]:
import os, gc, math, typing
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, Model, Input
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import cdist
from Data_preprocessing import xy_to_zone_vectorized

print("TF:", tf.__version__, "| GPUs:", tf.config.list_physical_devices('GPU'))

SEQ_LEN = 30
FPS = 10
HORIZON_SECONDS = 3
HORIZON_FRAMES = HORIZON_SECONDS * FPS
BATCH_SIZE = 64
LR = 1e-3
EPOCHS = 100
PATIENCE =10
N_ROWS, N_COLS = 3, 9
NUM_ZONES = N_ROWS * N_COLS
N_PLAYERS = 5
N_OPPONENTS = 5
TOTAL_NODES = N_PLAYERS + N_OPPONENTS
K_EDGES = 3

# ── Capacity & regularisation ───────────────────────────────────────────────
# train_acc < val_acc in previous run → over-regularised, not overfitting.
# Slightly restore capacity and ease dropout/L2.
GCN_HIDDEN = 48       # 32→48 (was over-shrunk)
GCN_OUT = 48          # 32→48
TCN_FILTERS = 96      # 64→96
DROPOUT = 0.25        # 0.4→0.25 (0.4 caused train_acc < val_acc)
WINDOW_STEP = 10      # stride between windows (was 1 — caused 47k overlapping seqs)
L2_REG = 5e-5         # 1e-4→5e-5 (ease weight penalty)

CO_ORDINATES = False

FEATURE_COLS = [
    "x_normalized", "y_normalized",
    "dx_avg_3", "dy_avg_3", "speed_normalized", "acceleration", "movement_angle",
    "acceleration_trend", "angle_change", "angle_stability", "speed_change_rate",
    "distance_from_center", "distance_from_goal_home", "distance_from_goal_away",
    "distance_from_sideline", "distance_to_ball", "ball_possession_proximity", "team_spread",
    "home_score", "away_score", "nearest_opponent_1", "nearest_opponent_2", "nearest_opponent_3",
    "defensive_line_distance", "team_centroid_distance", "position_encoded"
]

COORD_COLS = ("x_normalized", "y_normalized")

In [ ]:
train_df = pd.read_pickle('preprocessed_data/train_df.pkl')
val_df = pd.read_pickle('preprocessed_data/val_df.pkl')
test_df = pd.read_pickle('preprocessed_data/test_df.pkl')
df = pd.concat([train_df, val_df, test_df], ignore_index=True)

available = [c for c in FEATURE_COLS if c in df.columns]
missing = [c for c in FEATURE_COLS if c not in df.columns]
if missing:
    print(f"Missing features (removed): {missing}")
    FEATURE_COLS = available

scaler = StandardScaler()
scaler.fit(train_df[FEATURE_COLS].values)
print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,} | Features: {len(FEATURE_COLS)}")
print(f"Matches: {df['match_id'].nunique()}, Players: {df['player_id'].nunique()}")

In [ ]:
def _build_single_team(g, feature_cols, coord_cols, n_players):
    top_players = g["player_id"].value_counts().head(n_players).index.tolist()
    if len(top_players) < n_players:
        return None
    g = g[g["player_id"].isin(top_players)].copy()
    player_order = g.groupby("player_id")["x_normalized"].mean().sort_values().index.tolist()
    player_map = {p: i for i, p in enumerate(player_order)}
    g["node_idx"] = g["player_id"].map(player_map)
    g = g.sort_values(["frame_number", "node_idx"])
    fc = g.groupby("frame_number").size()
    valid_frames = sorted(fc[fc == n_players].index.tolist())
    g = g[g["frame_number"].isin(valid_frames)].sort_values(["frame_number", "node_idx"])
    n_frames = len(valid_frames)
    if n_frames < SEQ_LEN + HORIZON_FRAMES + 10:
        return None
    feats = np.nan_to_num(g[feature_cols].values, nan=0.0).reshape(n_frames, n_players, -1).astype(np.float32)
    coords = g[list(coord_cols)].values.reshape(n_frames, n_players, 2).astype(np.float32)
    frame_to_idx = {f: i for i, f in enumerate(valid_frames)}
    return {"features": feats, "coords": coords, "players": player_order, "frames": valid_frames, "frame_to_idx": frame_to_idx}

def build_team_tensors(df, feature_cols, coord_cols, n_players=N_PLAYERS, n_opponents=N_OPPONENTS):
    raw = {}
    for (mid, team), g in df.groupby(["match_id", "team_type"]):
        result = _build_single_team(g, feature_cols, coord_cols, n_players)
        if result is not None:
            raw[(mid, team)] = result

    matches = {}
    for (mid, team) in raw:
        matches.setdefault(mid, []).append(team)

    team_data = {}
    for mid, team_types in matches.items():
        for t in team_types:
            key = (mid, t)
            d = raw[key]
            opp_types = [x for x in team_types if x != t]
            if not opp_types or (mid, opp_types[0]) not in raw:
                n_frames = d["features"].shape[0]
                opp_feats = np.zeros((n_frames, n_opponents, len(feature_cols)), dtype=np.float32)
                opp_coords = np.zeros((n_frames, n_opponents, 2), dtype=np.float32)
                team_data[key] = {
                    "features": np.concatenate([d["features"], opp_feats], axis=1),
                    "coords": np.concatenate([d["coords"], opp_coords], axis=1),
                    "team_coords": d["coords"],
                    "players": d["players"]
                }
                continue

            opp_d = raw[(mid, opp_types[0])]
            common_frames = sorted(set(d["frames"]) & set(opp_d["frames"]))
            if len(common_frames) < SEQ_LEN + HORIZON_FRAMES + 10:
                continue

            t_idx = np.array([d["frame_to_idx"][f] for f in common_frames])
            o_idx = np.array([opp_d["frame_to_idx"][f] for f in common_frames])

            team_data[key] = {
                "features": np.concatenate([d["features"][t_idx], opp_d["features"][o_idx]], axis=1),
                "coords": np.concatenate([d["coords"][t_idx], opp_d["coords"][o_idx]], axis=1),
                "team_coords": d["coords"][t_idx],
                "players": d["players"]
            }
    return team_data

train_teams = build_team_tensors(train_df, FEATURE_COLS, COORD_COLS)
val_teams = build_team_tensors(val_df, FEATURE_COLS, COORD_COLS)
test_teams = build_team_tensors(test_df, FEATURE_COLS, COORD_COLS)
print(f"Graph teams -> Train: {len(train_teams)}, Val: {len(val_teams)}, Test: {len(test_teams)}")
for k, v in list(train_teams.items())[:3]:
    print(f"  {k}: {v['features'].shape[0]} frames, {v['features'].shape[1]} nodes ({N_PLAYERS}+{N_OPPONENTS}), {v['features'].shape[2]} feats")

In [ ]:
sample_key = list(train_teams.keys())[0]
sample = train_teams[sample_key]
coords = sample["team_coords"]

fig, axes = plt.subplots(2, 1, figsize=(16, 6), sharex=True)
for p in [0, 1, 2]:
    axes[0].plot(coords[:500, p, 0], label=f"Node {p}", alpha=0.8)
    axes[1].plot(coords[:500, p, 1], label=f"Node {p}", alpha=0.8)
axes[0].set_ylabel("x_normalized")
axes[0].legend()
axes[1].set_ylabel("y_normalized")
axes[1].set_xlabel("Frame")
fig.suptitle(f"Player position time series - {sample_key}")
plt.tight_layout()
plt.show()

In [ ]:
# EDA: Inter-player speed correlation heatmap (analogous to traffic route correlation)
speeds = np.sqrt(np.diff(coords[:, :, 0], axis=0)**2 + np.diff(coords[:, :, 1], axis=0)**2)
corr = np.corrcoef(speeds.T)

plt.figure(figsize=(8, 8))
plt.matshow(corr, 0, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar()
plt.xlabel("Player node")
plt.ylabel("Player node")
plt.title("Inter-player speed correlation")
plt.tight_layout()
plt.show()

In [ ]:
class GraphInfo:
    def __init__(self, edges, num_nodes):
        self.edges = edges
        self.num_nodes = num_nodes

def compute_adjacency_knn(positions, k=K_EDGES):
    n = len(positions)
    dists = cdist(positions, positions)
    adj = np.zeros((n, n), dtype=np.float32)
    for i in range(n):
        neighbors = np.argsort(dists[i])[1:k+1]
        adj[i, neighbors] = 1
        adj[neighbors, i] = 1
    np.fill_diagonal(adj, 0)
    return adj

all_avg_pos = [d["coords"].mean(axis=0) for d in train_teams.values()]
global_avg_pos = np.mean(all_avg_pos, axis=0)

adjacency_matrix = compute_adjacency_knn(global_avg_pos, k=K_EDGES)
node_idx, neighbor_idx = np.where(adjacency_matrix == 1)
graph = GraphInfo(edges=(node_idx.tolist(), neighbor_idx.tolist()), num_nodes=TOTAL_NODES)
print(f"Nodes: {graph.num_nodes}, Edges: {len(graph.edges[0])}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].matshow(adjacency_matrix, cmap="Blues")
axes[0].set_title("Adjacency Matrix")
axes[0].set_xlabel("Player node")
axes[0].set_ylabel("Player node")

team_colors = ["steelblue"] * N_PLAYERS + ["crimson"] * N_OPPONENTS
axes[1].scatter(global_avg_pos[:, 0], global_avg_pos[:, 1], s=200, c=team_colors, zorder=5)
for i in range(TOTAL_NODES):
    label = f"T{i}" if i < N_PLAYERS else f"O{i - N_PLAYERS}"
    axes[1].annotate(label, xy=(global_avg_pos[i, 0], global_avg_pos[i, 1]),
                     ha="center", va="center", fontsize=7, color="white", fontweight="bold")
for i, j in zip(node_idx, neighbor_idx):
    if i < j:
        axes[1].plot([global_avg_pos[i, 0], global_avg_pos[j, 0]],
                     [global_avg_pos[i, 1], global_avg_pos[j, 1]], "gray", alpha=0.5)
axes[1].set_title("Player Graph (teammates=blue, opponents=red)")
axes[1].set_xlabel("x_normalized")
axes[1].set_ylabel("y_normalized")
axes[1].set_aspect("equal")
plt.tight_layout()
plt.show()

In [ ]:
def count_sequences(team_dict):
    # Use WINDOW_STEP stride to avoid ~identical overlapping windows
    return sum(
        len(range(0, max(0, d["features"].shape[0] - SEQ_LEN - HORIZON_FRAMES + 1), WINDOW_STEP))
        for d in team_dict.values()
    )

def gnn_generator(team_dict, scaler, shuffle=False):
    keys = list(team_dict.keys())
    if shuffle:
        np.random.shuffle(keys)
    n_feat = len(FEATURE_COLS)
    for key in keys:
        data = team_dict[key]
        feats = data["features"].copy()
        team_coords = data["team_coords"]
        n_frames = feats.shape[0]
        flat = feats.reshape(-1, n_feat)
        feats = scaler.transform(flat).reshape(feats.shape)
        # stride = WINDOW_STEP to reduce correlated windows
        indices = list(range(0, n_frames - SEQ_LEN - HORIZON_FRAMES + 1, WINDOW_STEP))
        if shuffle:
            np.random.shuffle(indices)
        for i in indices:
            x = feats[i:i + SEQ_LEN]
            t_fut = i + SEQ_LEN - 1 + HORIZON_FRAMES
            if CO_ORDINATES:
                y = team_coords[t_fut].astype(np.float32)
            else:
                y = xy_to_zone_vectorized(team_coords[t_fut, :, 0], team_coords[t_fut, :, 1], N_ROWS, N_COLS).astype(np.int32)
            yield x, y

n_features = len(FEATURE_COLS)
if CO_ORDINATES:
    out_sig = (tf.TensorSpec((SEQ_LEN, TOTAL_NODES, n_features), tf.float32),
               tf.TensorSpec((N_PLAYERS, 2), tf.float32))
else:
    out_sig = (tf.TensorSpec((SEQ_LEN, TOTAL_NODES, n_features), tf.float32),
               tf.TensorSpec((N_PLAYERS,), tf.int32))

train_ds = tf.data.Dataset.from_generator(lambda: gnn_generator(train_teams, scaler, True), output_signature=out_sig)
val_ds = tf.data.Dataset.from_generator(lambda: gnn_generator(val_teams, scaler), output_signature=out_sig)
test_ds = tf.data.Dataset.from_generator(lambda: gnn_generator(test_teams, scaler), output_signature=out_sig)

train_seqs = count_sequences(train_teams)
val_seqs = count_sequences(val_teams)
test_seqs = count_sequences(test_teams)
train_steps = max(1, train_seqs // BATCH_SIZE)
val_steps = max(1, val_seqs // BATCH_SIZE)

train_ds = train_ds.shuffle(2048).batch(BATCH_SIZE, drop_remainder=True).prefetch(tf.data.AUTOTUNE).repeat()
val_ds = val_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print(f"Train: {train_seqs:,} seqs ({train_steps} steps/epoch) | Val: {val_seqs:,} | Test: {test_seqs:,}")
print(f"  (WINDOW_STEP={WINDOW_STEP} — reduced from stride-1 to cut correlated sequences)")


In [ ]:
class GraphConv(layers.Layer):
    def __init__(self, in_feat, out_feat, graph_info=None, aggregation="mean", l2_reg=0.0, **kwargs):
        super().__init__(**kwargs)
        self.in_feat = in_feat
        self.out_feat = out_feat
        self.aggregation = aggregation
        self.graph_info = graph_info
        self.l2_reg = l2_reg
        reg = tf.keras.regularizers.l2(l2_reg) if l2_reg > 0 else None
        self.w_self  = self.add_weight(shape=(in_feat, out_feat), initializer="glorot_uniform",
                                       regularizer=reg, name="w_self")
        self.w_neigh = self.add_weight(shape=(in_feat, out_feat), initializer="glorot_uniform",
                                       regularizer=reg, name="w_neigh")
        self.bias    = self.add_weight(shape=(out_feat,), initializer="zeros", name="bias")
        self.norm    = layers.LayerNormalization()

    def call(self, features):
        if self.graph_info is None:
            raise ValueError("graph_info must be provided")
        self_out = tf.matmul(features, self.w_self)
        neighbor_feats = tf.gather(features, self.graph_info.edges[1])
        agg_fn = tf.math.unsorted_segment_mean if self.aggregation == "mean" else tf.math.unsorted_segment_sum
        agg = agg_fn(neighbor_feats, self.graph_info.edges[0], self.graph_info.num_nodes)
        neigh_out = tf.matmul(agg, self.w_neigh)
        out = self_out + neigh_out + self.bias
        return tf.nn.relu(self.norm(out))

    def get_config(self):
        config = super().get_config()
        config.update({"in_feat": self.in_feat, "out_feat": self.out_feat,
                        "aggregation": self.aggregation, "l2_reg": self.l2_reg})
        return config


In [ ]:
from TCN import TCN

class TemporalC(layers.Layer):
    def __init__(self, in_feat, gcn_hidden, gcn_out, tcn_filters, graph_info,
                 output_dim=2, n_predict=N_PLAYERS, dropout=0.2, l2_reg=0.0, **kwargs):
        super().__init__(**kwargs)
        self.in_feat = in_feat
        self.gcn_hidden = gcn_hidden
        self.gcn_out = gcn_out
        self.tcn_filters = tcn_filters
        self.output_dim = output_dim
        self.n_predict = n_predict
        self.dropout_rate = dropout
        self.l2_reg = l2_reg
        self.graph_info = graph_info

        reg = tf.keras.regularizers.l2(l2_reg) if l2_reg > 0 else None

        self.gcn1 = GraphConv(in_feat, gcn_hidden, graph_info, l2_reg=l2_reg)
        self.gcn2 = GraphConv(gcn_hidden, gcn_out, graph_info, l2_reg=l2_reg)
        self.drop = layers.Dropout(dropout)

        self.tcn = TCN(
            nb_filters=tcn_filters,
            kernel_size=3,
            dilations=[1, 2, 4, 8, 16],
            padding="causal",
            return_sequences=False,
            dropout_rate=dropout
        )
        self.norm    = layers.LayerNormalization()
        self.dense1  = layers.Dense(tcn_filters // 2, activation="relu", kernel_regularizer=reg)
        self.dense_out = layers.Dense(output_dim, kernel_regularizer=reg)

    def call(self, inputs, training=False):
        # inputs: (batch, seq, nodes, feat)
        x = tf.transpose(inputs, [2, 0, 1, 3])   # (nodes, batch, seq, feat)
        x = self.gcn1(x)
        x = self.drop(x, training=training)
        x = self.gcn2(x)                           # (nodes, batch, seq, gcn_out)
        shape = tf.shape(x)
        # flatten nodes+batch for TCN: (nodes*batch, seq, gcn_out)
        x = tf.transpose(x, [1, 0, 2, 3])         # (batch, nodes, seq, gcn_out)
        x = tf.reshape(x, (shape[1] * shape[0], shape[2], shape[3]))
        x = self.tcn(x, training=training)         # (batch*nodes, gcn_out)
        x = self.norm(x)
        x = self.dense1(x)
        x = self.drop(x, training=training)
        x = self.dense_out(x)                      # (batch*nodes, output_dim)
        x = tf.reshape(x, (shape[1], shape[0], self.output_dim))  # (batch, nodes, output_dim)
        return x[:, :self.n_predict, :]            # (batch, N_PLAYERS, output_dim)

    def get_config(self):
        config = super().get_config()
        config.update({
            "in_feat": self.in_feat,
            "gcn_hidden": self.gcn_hidden,
            "gcn_out": self.gcn_out,
            "tcn_filters": self.tcn_filters,
            "output_dim": self.output_dim,
            "n_predict": self.n_predict,
            "dropout": self.dropout_rate,
            "l2_reg": self.l2_reg
        })
        return config


In [ ]:
# ── Reset model & optimizer before training ───────────────────────────────────
# REQUIRED: Adam optimizer carried LR=1e-5 and stale momentum from previous run.
# Rebuilding resets LR to LR=1e-3 and clears all optimizer state.
tf.keras.backend.clear_session()

output_dim = 2 if CO_ORDINATES else NUM_ZONES
inputs = Input(shape=(SEQ_LEN, TOTAL_NODES, n_features), name="graph_input")
x = TemporalC(
    in_feat=n_features,
    gcn_hidden=GCN_HIDDEN,
    gcn_out=GCN_OUT,
    tcn_filters=TCN_FILTERS,
    graph_info=graph,
    output_dim=output_dim,
    n_predict=N_PLAYERS,
    dropout=DROPOUT,
    l2_reg=L2_REG
)(inputs)
model = Model(inputs, x, name="GNN_Temporal")

if CO_ORDINATES:
    model.compile(optimizer=tf.keras.optimizers.Adam(LR), loss="mse", metrics=["mae"])
else:
    model.compile(
        optimizer=tf.keras.optimizers.Adam(LR),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=[
            "sparse_categorical_accuracy",
            tf.keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3_acc"),
            tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5, name="top5_acc")
        ],
    )

print(f"Model reset. LR={LR}, dropout={DROPOUT}, label_smoothing=0.05, L2={L2_REG}")
model.summary()

In [ ]:
import gc
class GarbageCollectorCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        gc.collect() # Resets the global state (useful for heavy models)

In [ ]:
# # ── Class weights for zone imbalance ─────────────────────────────────────────
# from collections import Counter
# from sklearn.utils.class_weight import compute_class_weight

# zone_counts = Counter()
# for d in train_teams.values():
#     tc = d["team_coords"]
#     n_frames = tc.shape[0]
#     for i in range(0, n_frames - SEQ_LEN - HORIZON_FRAMES + 1, WINDOW_STEP):
#         t_fut = i + SEQ_LEN - 1 + HORIZON_FRAMES
#         zones = xy_to_zone_vectorized(tc[t_fut, :, 0], tc[t_fut, :, 1], N_ROWS, N_COLS)
#         for z in zones:
#             zone_counts[int(z)] += 1

# all_zone_labels = []
# for z, cnt in zone_counts.items():
#     all_zone_labels.extend([z] * cnt)
# all_zone_labels = np.array(all_zone_labels)

# weights = compute_class_weight("balanced", classes=np.arange(NUM_ZONES), y=all_zone_labels)
# class_weight_dict = {i: float(w) for i, w in enumerate(weights)}

# print("Zone class weight grid (N_ROWS x N_COLS):")
# w_grid = np.array([class_weight_dict[z] for z in range(NUM_ZONES)]).reshape(N_ROWS, N_COLS)
# print(np.round(w_grid, 2))


In [57]:
monitor = "val_mae" if CO_ORDINATES else "val_sparse_categorical_accuracy"
mode = "min" if CO_ORDINATES else "max"
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor=monitor, patience=PATIENCE,
                                     restore_best_weights=True, mode=mode),
    tf.keras.callbacks.ModelCheckpoint("best_gnn_lstm", monitor=monitor,
                                       save_best_only=True, mode=mode,
                                       save_weights_only=True),
    # patience=6 so LR only drops after 6 stagnant epochs — prevents LR
    # collapsing too early (was patience=3 which dropped LR 3x in 12 epochs)
    tf.keras.callbacks.ReduceLROnPlateau(monitor=monitor, factor=0.5,
                                         patience=6, min_lr=1e-6, mode=mode,
                                         verbose=1),
    tf.keras.callbacks.CSVLogger('training_history.csv', separator=',', append=False),
    GarbageCollectorCallback()
]

# class_weight REMOVED — it inflated training loss to ~60 (vs val ~2.2),
# making the gradient signal incoherent and training accuracy worse than val.
# Label smoothing in the loss handles class imbalance more gently.

history = model.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS,
    steps_per_epoch=train_steps, callbacks=callbacks,
    validation_steps=val_steps
)


74/74 [==============================] - 15s 197ms/step - loss: 1.9403 - sparse_categorical_accuracy: 0.3947 - top3_acc: 0.7051 - top5_acc: 0.8242 - val_loss: 1.9045 - val_sparse_categorical_accuracy: 0.4043 - val_top3_acc: 0.7474 - val_top5_acc: 0.8592 - lr: 0.0010
Epoch 15/100
74/74 [==============================] - 15s 190ms/step - loss: 1.9352 - sparse_categorical_accuracy: 0.3933 - top3_acc: 0.7072 - top5_acc: 0.8239 - val_loss: 1.9044 - val_sparse_categorical_accuracy: 0.4011 - val_top3_acc: 0.7434 - val_top5_acc: 0.8594 - lr: 0.0010
Epoch 16/100
74/74 [==============================] - ETA: 0s - loss: 1.9177 - sparse_categorical_accuracy: 0.3981 - top3_acc: 0.7103 - top5_acc: 0.8300
Epoch 16: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
74/74 [==============================] - 15s 187ms/step - loss: 1.9177 - sparse_categorical_accuracy: 0.3981 - top3_acc: 0.7103 - top5_acc: 0.8300 - val_loss: 1.9089 - val_sparse_categorical_accuracy: 0.3989 - val_top3_acc:

In [ ]:
# import h5py

# f = h5py.File("best_gnn_lstm.weights.h5", "r")
# print(list(f.keys()))

In [ ]:
# model.load_weights(
#     "best_gnn_lstm.weights.h5",by_name=True,skip_mismatch=True
    
# )


In [ ]:
# if CO_ORDINATES:
#     model.compile(
#         optimizer=tf.keras.optimizers.Adam(LR),
#         loss="mse",
#         metrics=["mae"]
#     )
# else:
#     model.compile(
#         optimizer=tf.keras.optimizers.Adam(LR),
#         loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
#         metrics=[
#             "sparse_categorical_accuracy",
#             tf.keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3_acc"),
#             tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5, name="top5_acc")
#         ]
#     )

In [ ]:
results = model.evaluate(test_ds)
print(f"\nTest: {dict(zip(model.metrics_names, results))}")

naive_errs = []
for d in test_teams.values():
    c = d["team_coords"]
    for i in range(c.shape[0] - SEQ_LEN - HORIZON_FRAMES + 1):
        naive_errs.append(np.abs(c[i + SEQ_LEN - 1] - c[i + SEQ_LEN - 1 + HORIZON_FRAMES]).mean())
print(f"Naive MAE (last-position baseline): {np.mean(naive_errs):.4f}")
if CO_ORDINATES:
    print(f"Model MAE: {results[1]:.4f}")

In [ ]:
# Forecast visualization: predicted vs actual for select players
for x_batch, y_batch in test_ds.take(2):
    y_pred = model.predict(x_batch)

if CO_ORDINATES:
    n_show = min(100, len(y_batch))
    fig, axes = plt.subplots(2, 1, figsize=(16, 8))
    for p in [0, 5, 10]:
        axes[0].plot(y_batch[:n_show, p, 0].numpy(), label=f"P{p} actual", alpha=0.7)
        axes[0].plot(y_pred[:n_show, p, 0], '--', label=f"P{p} pred", alpha=0.7)
        axes[1].plot(y_batch[:n_show, p, 1].numpy(), label=f"P{p} actual", alpha=0.7)
        axes[1].plot(y_pred[:n_show, p, 1], '--', label=f"P{p} pred", alpha=0.7)
    axes[0].set_ylabel("x_normalized")
    axes[0].legend(ncol=3, fontsize=8)
    axes[1].set_ylabel("y_normalized")
    axes[1].set_xlabel("Sample")
    fig.suptitle("GNN-LSTM: Predictions vs Actual")
    plt.tight_layout()
    plt.show()

    # Per-node MAE bar chart
    node_mae = np.abs(y_pred - y_batch.numpy()).mean(axis=0)
    fig, ax = plt.subplots(figsize=(10, 4))
    x_pos = np.arange(N_PLAYERS)
    ax.bar(x_pos - 0.15, node_mae[:, 0], 0.3, label="MAE x")
    ax.bar(x_pos + 0.15, node_mae[:, 1], 0.3, label="MAE y")
    ax.set_xlabel("Player Node")
    ax.set_ylabel("MAE")
    ax.set_title("Per-Player Prediction Error")
    ax.legend()
    ax.set_xticks(x_pos)
    plt.tight_layout()
    plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np



# take one batch
for x_batch, y_batch in test_ds.take(1):
    y_pred = model.predict(x_batch)

if CO_ORDINATES:
    n_show = min(100, len(y_batch))
    fig, axes = plt.subplots(2, 1, figsize=(16, 8))
    for p in range(min(1, N_PLAYERS)):  # show up to 5 players
        axes[0].plot(y_batch[:n_show, p, 0].numpy(), label=f"P{p} actual", alpha=0.7)
        axes[0].plot(y_pred[:n_show, p, 0], '--', label=f"P{p} pred", alpha=0.7)
        axes[1].plot(y_batch[:n_show, p, 1].numpy(), label=f"P{p} actual", alpha=0.7)
        axes[1].plot(y_pred[:n_show, p, 1], '--', label=f"P{p} pred", alpha=0.7)
    axes[0].set_ylabel("x_normalized")
    axes[0].legend(ncol=3, fontsize=8)
    axes[1].set_ylabel("y_normalized")
    axes[1].set_xlabel("Sample")
    fig.suptitle("GNN-LSTM: Predictions vs Actual")
    plt.tight_layout()
    plt.show()

    # Per-node MAE
    node_mae = np.abs(y_pred - y_batch.numpy()).mean(axis=0)
    fig, ax = plt.subplots(figsize=(10, 4))
    x_pos = np.arange(N_PLAYERS)
    ax.bar(x_pos - 0.15, node_mae[:, 0], 0.3, label="MAE x")
    ax.bar(x_pos + 0.15, node_mae[:, 1], 0.3, label="MAE y")
    ax.set_xlabel("Player Node")
    ax.set_ylabel("MAE")
    ax.set_title("Per-Player Prediction Error")
    ax.legend()
    ax.set_xticks(x_pos)
    plt.tight_layout()
    plt.show()

In [ ]:
print("Available (match_id, team) keys in test set:")
for i, (mid, team) in enumerate(test_teams.keys()):
    n_frames = test_teams[(mid, team)]["features"].shape[0]
    max_sec = (n_frames - HORIZON_FRAMES) / FPS
    print(f"  [{i:>3}]  match={mid}  team={team}  frames={n_frames}  queryable up to {max_sec:.1f}s")

In [ ]:
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

# ── CONFIGURE HERE ───────────────────────────────────────────────────────────
SELECT_MATCH   = list(test_teams.keys())[0][0]   # match_id  — change to any test match
SELECT_TEAM    = list(test_teams.keys())[0][1]   # team_type — change to the team within the match
SELECT_PLAYER  = 0                                # node index 0-4 (sorted by avg x position)
SELECT_SECOND  = 60                               # query time in seconds from match start
TOP_K          = 5
# ─────────────────────────────────────────────────────────────────────────────

def predict_player_at_time(match_id, team, player_idx, query_second,
                           test_teams, model, scaler, top_k=TOP_K):
    key = (match_id, team)
    if key not in test_teams:
        raise ValueError(f"Key {key} not found. Available: {list(test_teams.keys())[:5]}")

    data       = test_teams[key]
    feats_all  = data["features"]          # (n_frames, TOTAL_NODES, n_feat)
    team_coords = data["team_coords"]       # (n_frames, N_PLAYERS, 2)
    n_frames   = feats_all.shape[0]
    n_feat     = feats_all.shape[-1]

    query_frame = int(query_second * FPS)
    query_frame = min(max(query_frame, SEQ_LEN), n_frames - HORIZON_FRAMES - 1)

    x_raw  = feats_all[query_frame - SEQ_LEN : query_frame]   # (SEQ_LEN, TOTAL_NODES, n_feat)
    x_scaled = scaler.transform(x_raw.reshape(-1, n_feat)).reshape(x_raw.shape)
    x_in   = x_scaled[np.newaxis]                              # (1, SEQ_LEN, TOTAL_NODES, n_feat)

    logits = model.predict(x_in, verbose=0)                   # (1, N_PLAYERS, NUM_ZONES)
    probs  = tf.nn.softmax(logits[0, player_idx]).numpy()      # (NUM_ZONES,)
    top_indices = np.argsort(probs)[::-1][:top_k]

    true_frame   = query_frame + HORIZON_FRAMES - 1
    true_x = team_coords[true_frame, player_idx, 0]
    true_y = team_coords[true_frame, player_idx, 1]
    true_zone = int(xy_to_zone_vectorized(
        np.array([true_x]), np.array([true_y]), N_ROWS, N_COLS)[0])

    current_x = team_coords[query_frame - 1, player_idx, 0]
    current_y = team_coords[query_frame - 1, player_idx, 1]

    return {
        "probs": probs,
        "top_indices": top_indices,
        "true_zone": true_zone,
        "true_x": float(true_x),
        "true_y": float(true_y),
        "current_x": float(current_x),
        "current_y": float(current_y),
        "query_frame": query_frame,
        "n_frames": n_frames
    }


def plot_zone_prediction(result, match_id, team, player_idx, query_second, top_k=TOP_K):
    probs       = result["probs"]
    true_zone   = result["true_zone"]
    top_indices = result["top_indices"]

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # ── Left: Zone probability heatmap ───────────────────────────────────────
    ax = axes[0]
    grid = probs.reshape(N_ROWS, N_COLS)
    im = ax.imshow(grid, cmap="YlOrRd", vmin=0, vmax=probs.max(), aspect="auto",
                   origin="upper", extent=[0, 1, 1, 0])
    plt.colorbar(im, ax=ax, label="Probability")

    cell_w = 1.0 / N_COLS
    cell_h = 1.0 / N_ROWS
    for z in range(N_ROWS * N_COLS):
        row, col = divmod(z, N_COLS)
        cx = col * cell_w + cell_w / 2
        cy = row * cell_h + cell_h / 2
        ax.text(cx, cy, f"{z}", ha="center", va="center", fontsize=7,
                color="black", alpha=0.6)

    true_row, true_col = divmod(true_zone, N_COLS)
    tx_c = true_col * cell_w + cell_w / 2
    ty_c = true_row * cell_h + cell_h / 2
    ax.scatter([result["current_x"]], [result["current_y"]], marker="o",
               s=150, c="blue", zorder=5, label="Current pos")
    ax.scatter([result["true_x"]], [result["true_y"]], marker="X",
               s=200, c="lime", zorder=5, label="Actual +3s")
    ax.scatter([tx_c], [ty_c], marker="s", s=300, facecolors="none",
               edgecolors="lime", linewidths=2, zorder=5, label=f"True zone {true_zone}")
    ax.set_xlim(0, 1); ax.set_ylim(1, 0)
    ax.set_xlabel("x_normalized"); ax.set_ylabel("y_normalized")
    ax.set_title(f"Zone Probability Heatmap\n{match_id} | {team} | Player-{player_idx} | t={query_second}s")
    ax.legend(fontsize=8, loc="upper right")

    # ── Right: Top-K bar chart ────────────────────────────────────────────────
    ax2 = axes[1]
    colors = ["#e74c3c" if i == true_zone else "#3498db" for i in top_indices]
    bars = ax2.barh(range(top_k), probs[top_indices][::-1], color=colors[::-1])
    ax2.set_yticks(range(top_k))
    ax2.set_yticklabels([f"Zone {i}" for i in top_indices[::-1]])
    ax2.set_xlabel("Probability")
    ax2.set_title(f"Top-{top_k} Predicted Zones\n(red = true zone)")
    for i, (bar, idx) in enumerate(zip(bars, top_indices[::-1])):
        ax2.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height() / 2,
                 f"{probs[idx]:.3f}", va="center", fontsize=9)
    hit_1 = top_indices[0] == true_zone
    hit_k = true_zone in top_indices
    ax2.set_title(f"Top-{top_k} Predicted Zones  |  Top-1: {'✓' if hit_1 else '✗'}  Top-{top_k}: {'✓' if hit_k else '✗'}")

    plt.tight_layout()
    plt.show()

    print(f"Query  : match={match_id}, team={team}, player_idx={player_idx}, "
          f"t={query_second}s (frame {result['query_frame']}/{result['n_frames']})")
    print(f"Top-1 predicted zone : {top_indices[0]}  (p={probs[top_indices[0]]:.4f})")
    print(f"True zone (+3s)       : {true_zone}  (p={probs[true_zone]:.4f})")
    print(f"Top-1 correct         : {hit_1}")
    print(f"Top-{top_k} correct       : {hit_k}")


result = predict_player_at_time(
    SELECT_MATCH, SELECT_TEAM, SELECT_PLAYER, SELECT_SECOND,
    test_teams, model, scaler, top_k=TOP_K
)
plot_zone_prediction(result, SELECT_MATCH, SELECT_TEAM, SELECT_PLAYER, SELECT_SECOND, top_k=TOP_K)